<a href="https://colab.research.google.com/github/maneeha/KGLLM/blob/main/Eunning_GraphRAG_v2_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GraphRAG Implementation with LlamaIndex - V2

[GraphRAG (Graphs + Retrieval Augmented Generation)](https://www.microsoft.com/en-us/research/project/graphrag/) combines the strengths of Retrieval Augmented Generation (RAG) and Query-Focused Summarization (QFS) to effectively handle complex queries over large text datasets. While RAG excels in fetching precise information, it struggles with broader queries that require thematic understanding, a challenge that QFS addresses but cannot scale well. GraphRAG integrates these approaches to offer responsive and thorough querying capabilities across extensive, diverse text corpora.

This notebook provides guidance on constructing the GraphRAG pipeline using the LlamaIndex PropertyGraph abstractions using Neo4J.

This notebook updates the GraphRAG pipeline to v2. If you haven’t checked v1 yet, you can find it [here](https://github.com/run-llama/llama_index/blob/main/docs/docs/examples/cookbooks/GraphRAG_v1.ipynb). Following are the updates to the existing implementation:

1. Integrate with Neo4J Graph database.
2. Embedding based retrieval.



## Installation

`graspologic` is used to use hierarchical_leiden for building communities.

In [ ]:
!pip install llama-index llama-index-graph-stores-neo4j graspologic numpy==1.24.4 scipy==1.12.0 future -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 71.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 4.2 MB/s eta 0:00:00
   ━━━

In [ ]:
import os
import pandas as pd
from llama_index.core import Document


In [ ]:
!pip install pdfplumber


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 99.4 MB/s eta 0:00:00


In [ ]:
import pdfplumber


## Load Data

We will use a sample news article dataset retrieved from Diffbot, which Tomaz has conveniently made available on GitHub for easy access.

The dataset contains 2,500 samples; for ease of experimentation, we will use 50 of these samples, which include the `title` and `text` of news articles.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import pdfplumber
from tqdm import tqdm

# Initialize an empty list to hold the documents
documents = []

# Specify the directory containing your PDF files
pdf_dir = '/content/drive/MyDrive/refined/content/test1'

# Get a list of PDF files in the directory
pdf_files = [filename for filename in os.listdir(pdf_dir) if filename.endswith('.pdf')]

# Loop through all PDF files in the specified directory with tqdm
for filename in tqdm(pdf_files, desc="Processing PDF files"):
    pdf_path = os.path.join(pdf_dir, filename)

    # Open and read the PDF file
    with pdfplumber.open(pdf_path) as pdf:
        for page in tqdm(pdf.pages, desc=f"Processing pages of {filename}", leave=False):
            text = page.extract_text()
            if text:  # Check if text was extracted
                documents.append(Document(text=text))

Processing PDF files: 100%|██████████| 2/2 [00:04<00:00,  2.32s/it]


Prepare documents as required by LlamaIndex

In [ ]:
len(documents)

13

In [ ]:
documents[1]

Document(id_='1fb7e3b3-92cf-40a5-8797-4ddd1b9853c5', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text="P.J. Burrows, P.J. Gray, A-L. Kinmonth, et al. Original papers\nResults The age distribution at diagnosis of patients in each treat-\nment group is shown in Figure l(b). Although non-insulin-\nPrevalence\ndependentdiabeteswasusuallydiagnosedinlatemiddleageand\nA total of 431 diabetic patients were identified, giving a insulin-dependent diabetes in the young, 110o ofnon-insulin-\nprevalenceof0.85%o.Amongindividualpractices,theprevalence\ndependentdiabeticspresentedundertheageof40yearsand 10%\nofdiabetes varied from 0.6%/o to 1.2%, being greatest in those ofinsulin-dependent diabeticspresentedovertheageof50years.\npractices withthegreatestproportion ofpatients over75years\nof age. There was a slight excess

## Setup API Key and LLM

In [ ]:
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjit

In [ ]:
# from llama_index.llms.ollama import Ollama
# llm = Ollama(model="llama3.2", request_timeout=100000.0)

In [ ]:
# from llama_index.embeddings.ollama import OllamaEmbedding

# embed_model = OllamaEmbedding(
#     model_name="nomic-embed-text:latest",
#     base_url="http://localhost:11434",
#     ollama_additional_kwargs={"mirostat": 0},
# )

In [ ]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device='cpu'  # Change 'cuda' to 'cpu'
)

NameError: name 'HuggingFaceEmbedding' is not defined

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, SimpleDirectoryReader, PropertyGraphIndex
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

# Configure Qwen2.5-7B-Instruct model
model_name = "Qwen/Qwen2.5-7B-Instruct"

llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    max_new_tokens=1024,
    model_kwargs={
        "torch_dtype": "auto",
        "trust_remote_code": True
    },
    tokenizer_kwargs={
        "trust_remote_code": True,
        "pad_token": "<|endoftext|>"
    },
    generate_kwargs={
        "do_sample": True,
        "temperature": 0.7,
        "repetition_penalty": 1.1
    },
    # Qwen2.5 specific chat template formatting
    query_wrapper_prompt="""<|im_start|>system
    You are a helpful assistant.<|im_end|>
<|im_start|>user
{query_str}<|im_end|>
<|im_start|>assistant"""
)

# Keep the rest of your existing configuration
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device='cuda',
)

Settings.llm = llm
Settings.embed_model = embed_model

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
!pip install llama-index-embeddings-huggingface

In [ ]:
!pip install llama-index


In [ ]:
!pip install --upgrade llama-index



In [ ]:
!pip install llama-index-llms-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 762.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [ ]:
import torch
print(torch.cuda.is_available())


False


In [ ]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device='cpu'  # Change 'cuda' to 'cpu'
)


In [ ]:
#from llama_index.llms.huggingface import HuggingFaceLLM


In [ ]:
llm.complete("Hello").text

' Hello! How can I assist you today?'

In [ ]:
from llama_index.core import Settings
Settings.llm = llm
Settings.embed_model = embed_model


## GraphRAGExtractor

The GraphRAGExtractor class is designed to extract triples (subject-relation-object) from text and enrich them by adding descriptions for entities and relationships to their properties using an LLM.

This functionality is similar to that of the `SimpleLLMPathExtractor`, but includes additional enhancements to handle entity, relationship descriptions. For guidance on implementation, you may look at similar existing [extractors](https://docs.llamaindex.ai/en/latest/examples/property_graph/Dynamic_KG_Extraction/?h=comparing).

Here's a breakdown of its functionality:

**Key Components:**

1. `llm:` The language model used for extraction.
2. `extract_prompt:` A prompt template used to guide the LLM in extracting information.
3. `parse_fn:` A function to parse the LLM's output into structured data.
4. `max_paths_per_chunk:` Limits the number of triples extracted per text chunk.
5. `num_workers:` For parallel processing of multiple text nodes.


**Main Methods:**

1. `__call__:` The entry point for processing a list of text nodes.
2. `acall:` An asynchronous version of __call__ for improved performance.
3. `_aextract:` The core method that processes each individual node.


**Extraction Process:**

For each input node (chunk of text):
1. It sends the text to the LLM along with the extraction prompt.
2. The LLM's response is parsed to extract entities, relationships, descriptions for entities and relations.
3. Entities are converted into EntityNode objects. Entity description is stored in metadata
4. Relationships are converted into Relation objects. Relationship description is stored in metadata.
5. These are added to the node's metadata under KG_NODES_KEY and KG_RELATIONS_KEY.

**NOTE:** In the current implementation, we are using only relationship descriptions. In the next implementation, we will utilize entity descriptions during the retrieval stage.

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

from typing import Any, List, Callable, Optional, Union, Dict
from IPython.display import Markdown, display

from llama_index.core.async_utils import run_jobs
from llama_index.core.indices.property_graph.utils import (
    default_parse_triplets_fn,
)
from llama_index.core.graph_stores.types import (
    EntityNode,
    KG_NODES_KEY,
    KG_RELATIONS_KEY,
    Relation,
)
from llama_index.core.llms.llm import LLM
from llama_index.core.prompts import PromptTemplate
from llama_index.core.prompts.default_prompts import (
    DEFAULT_KG_TRIPLET_EXTRACT_PROMPT,
)
from llama_index.core.schema import TransformComponent, BaseNode
from llama_index.core.bridge.pydantic import BaseModel, Field


class GraphRAGExtractor(TransformComponent):
    """Extract triples from a graph.

    Uses an LLM and a simple prompt + output parsing to extract paths (i.e. triples) and entity, relation descriptions from text.

    Args:
        llm (LLM):
            The language model to use.
        extract_prompt (Union[str, PromptTemplate]):
            The prompt to use for extracting triples.
        parse_fn (callable):
            A function to parse the output of the language model.
        num_workers (int):
            The number of workers to use for parallel processing.
        max_paths_per_chunk (int):
            The maximum number of paths to extract per chunk.
    """

    llm: LLM
    extract_prompt: PromptTemplate
    parse_fn: Callable
    num_workers: int
    max_paths_per_chunk: int

    def __init__(
        self,
        llm: Optional[LLM] = None,
        extract_prompt: Optional[Union[str, PromptTemplate]] = None,
        parse_fn: Callable = default_parse_triplets_fn,
        max_paths_per_chunk: int = 10,
        num_workers: int = 4,
    ) -> None:
        """Init params."""
        from llama_index.core import Settings

        if isinstance(extract_prompt, str):
            extract_prompt = PromptTemplate(extract_prompt)

        super().__init__(
            llm=llm or Settings.llm,
            extract_prompt=extract_prompt or DEFAULT_KG_TRIPLET_EXTRACT_PROMPT,
            parse_fn=parse_fn,
            num_workers=num_workers,
            max_paths_per_chunk=max_paths_per_chunk,
        )

    @classmethod
    def class_name(cls) -> str:
        return "GraphExtractor"

    def __call__(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]:
        """Extract triples from nodes."""
        return asyncio.run(
            self.acall(nodes, show_progress=show_progress, **kwargs)
        )

    async def _aextract(self, node: BaseNode) -> BaseNode:
        """Extract triples from a node."""
        assert hasattr(node, "text")

        text = node.get_content(metadata_mode="llm")
        try:
            llm_response = await self.llm.apredict(
                self.extract_prompt,
                text=text,
                max_knowledge_triplets=self.max_paths_per_chunk,
            )
            entities, entities_relationship = self.parse_fn(llm_response)
        except ValueError:
            entities = []
            entities_relationship = []

        existing_nodes = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        entity_metadata = node.metadata.copy()
        for entity, entity_type, description in entities:
            entity_metadata["entity_description"] = description
            entity_node = EntityNode(
                name=entity, label=entity_type, properties=entity_metadata
            )
            existing_nodes.append(entity_node)

        relation_metadata = node.metadata.copy()
        for triple in entities_relationship:
            subj, obj, rel, description = triple
            relation_metadata["relationship_description"] = description
            rel_node = Relation(
                label=rel,
                source_id=subj,
                target_id=obj,
                properties=relation_metadata,
            )

            existing_relations.append(rel_node)

        node.metadata[KG_NODES_KEY] = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        return node

    async def acall(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]:
        """Extract triples from nodes async."""
        jobs = []
        for node in nodes:
            jobs.append(self._aextract(node))

        return await run_jobs(
            jobs,
            workers=self.num_workers,
            show_progress=show_progress,
            desc="Extracting paths from text",
        )

## GraphRAGStore

The `GraphRAGStore` class is an extension of the `Neo4jPropertyGraphStore`class, designed to implement GraphRAG pipeline. Here's a breakdown of its key components and functions:


The class uses community detection algorithms to group related nodes in the graph and then it generates summaries for each community using an LLM.


**Key Methods:**

`build_communities():`

1. Converts the internal graph representation to a NetworkX graph.

2. Applies the hierarchical Leiden algorithm for community detection.

3. Collects detailed information about each community.

4. Generates summaries for each community.

`generate_community_summary(text):`

1. Uses LLM to generate a summary of the relationships in a community.
2. The summary includes entity names and a synthesis of relationship descriptions.

`_create_nx_graph():`

1. Converts the internal graph representation to a NetworkX graph for community detection.

`_collect_community_info(nx_graph, clusters):`

1. Collects detailed information about each node based on its community.
2. Creates a string representation of each relationship within a community.

`_summarize_communities(community_info):`

1. Generates and stores summaries for each community using LLM.

`get_community_summaries():`

1. Returns the community summaries by building them if not already done.

In [ ]:
import re
import networkx as nx
from graspologic.partition import hierarchical_leiden
from collections import defaultdict

from llama_index.core.llms import ChatMessage
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore


class GraphRAGStore(Neo4jPropertyGraphStore):
    community_summary = {}
    entity_info = None
    max_cluster_size = 5

    def generate_community_summary(self, text):
        """Generate summary for a given text using an LLM."""
        messages = [
            ChatMessage(
                role="system",
                content=(
                    "You are provided with a set of relationships from a knowledge graph, each represented as "
                    "entity1->entity2->relation->relationship_description. Your task is to create a summary of these "
                    "relationships. The summary should include the names of the entities involved and a concise synthesis "
                    "of the relationship descriptions. The goal is to capture the most critical and relevant details that "
                    "highlight the nature and significance of each relationship. Ensure that the summary is coherent and "
                    "integrates the information in a way that emphasizes the key aspects of the relationships."
                ),
            ),
            ChatMessage(role="user", content=text),
        ]
        response = llm.chat(messages)
        clean_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return clean_response

    def build_communities(self):
        """Builds communities from the graph and summarizes them."""
        nx_graph = self._create_nx_graph()
        community_hierarchical_clusters = hierarchical_leiden(
            nx_graph, max_cluster_size=self.max_cluster_size
        )
        self.entity_info, community_info = self._collect_community_info(
            nx_graph, community_hierarchical_clusters
        )
        self._summarize_communities(community_info)

    def _create_nx_graph(self):
        """Converts internal graph representation to NetworkX graph."""
        nx_graph = nx.Graph()
        triplets = self.get_triplets()
        for entity1, relation, entity2 in triplets:
            nx_graph.add_node(entity1.name)
            nx_graph.add_node(entity2.name)
            nx_graph.add_edge(
                relation.source_id,
                relation.target_id,
                relationship=relation.label,
                description=relation.properties["relationship_description"],
            )
        return nx_graph

    def _collect_community_info(self, nx_graph, clusters):
        """
        Collect information for each node based on their community,
        allowing entities to belong to multiple clusters.
        """
        entity_info = defaultdict(set)
        community_info = defaultdict(list)

        for item in clusters:
            node = item.node
            cluster_id = item.cluster

            # Update entity_info
            entity_info[node].add(cluster_id)

            for neighbor in nx_graph.neighbors(node):
                edge_data = nx_graph.get_edge_data(node, neighbor)
                if edge_data:
                    detail = f"{node} -> {neighbor} -> {edge_data['relationship']} -> {edge_data['description']}"
                    community_info[cluster_id].append(detail)

        # Convert sets to lists for easier serialization if needed
        entity_info = {k: list(v) for k, v in entity_info.items()}

        return dict(entity_info), dict(community_info)

    def _summarize_communities(self, community_info):
        """Generate and store summaries for each community."""
        for community_id, details in community_info.items():
            details_text = (
                "\n".join(details) + "."
            )  # Ensure it ends with a period
            self.community_summary[
                community_id
            ] = self.generate_community_summary(details_text)

    def get_community_summaries(self):
        """Returns the community summaries, building them if not already done."""
        if not self.community_summary:
            self.build_communities()
        return self.community_summary

## GraphRAGQueryEngine

The GraphRAGQueryEngine class is a custom query engine designed to process queries using the GraphRAG approach. It leverages the community summaries generated by the GraphRAGStore to answer user queries. Here's a breakdown of its functionality:

**Main Components:**

`graph_store:` An instance of GraphRAGStore, which contains the community summaries.
`llm:` A Language Model (LLM) used for generating and aggregating answers.


**Key Methods:**

`custom_query(query_str: str)`

1. This is the main entry point for processing a query. It retrieves community summaries, generates answers from each summary, and then aggregates these answers into a final response.

`generate_answer_from_summary(community_summary, query):`

1. Generates an answer for the query based on a single community summary.
Uses the LLM to interpret the community summary in the context of the query.

`aggregate_answers(community_answers):`

1. Combines individual answers from different communities into a coherent final response.
2. Uses the LLM to synthesize multiple perspectives into a single, concise answer.


**Query Processing Flow:**

1. Retrieve community summaries from the graph store.
2. For each community summary, generate a specific answer to the query.
3. Aggregate all community-specific answers into a final, coherent response.


**Example usage:**

```
query_engine = GraphRAGQueryEngine(graph_store=graph_store, llm=llm)

response = query_engine.query("query")
```

In [ ]:
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.llms import LLM
from llama_index.core import PropertyGraphIndex

import re


class GraphRAGQueryEngine(CustomQueryEngine):
    graph_store: GraphRAGStore
    index: PropertyGraphIndex
    llm: LLM
    similarity_top_k: int = 20

    def custom_query(self, query_str: str) -> str:
        """Process all community summaries to generate answers to a specific query."""

        entities = self.get_entities(query_str, self.similarity_top_k)

        community_ids = self.retrieve_entity_communities(
            self.graph_store.entity_info, entities
        )
        community_summaries = self.graph_store.get_community_summaries()
        community_answers = [
            self.generate_answer_from_summary(community_summary, query_str)
            for id, community_summary in community_summaries.items()
            if id in community_ids
        ]

        final_answer = self.aggregate_answers(community_answers)
        return final_answer

    def get_entities(self, query_str, similarity_top_k):
        nodes_retrieved = self.index.as_retriever(
            similarity_top_k=similarity_top_k
        ).retrieve(query_str)

        enitites = set()
        pattern = (
            r"^(\w+(?:\s+\w+)*)\s*->\s*([a-zA-Z\s]+?)\s*->\s*(\w+(?:\s+\w+)*)$"
        )

        for node in nodes_retrieved:
            matches = re.findall(
                pattern, node.text, re.MULTILINE | re.IGNORECASE
            )

            for match in matches:
                subject = match[0]
                obj = match[2]
                enitites.add(subject)
                enitites.add(obj)

        return list(enitites)

    def retrieve_entity_communities(self, entity_info, entities):
        """
        Retrieve cluster information for given entities, allowing for multiple clusters per entity.

        Args:
        entity_info (dict): Dictionary mapping entities to their cluster IDs (list).
        entities (list): List of entity names to retrieve information for.

        Returns:
        List of community or cluster IDs to which an entity belongs.
        """
        community_ids = []

        for entity in entities:
            if entity in entity_info:
                community_ids.extend(entity_info[entity])

        return list(set(community_ids))

    def generate_answer_from_summary(self, community_summary, query):
        """Generate an answer from a community summary based on a given query using LLM."""
        prompt = (
            f"Given the community summary: {community_summary}, "
            f"how would you answer the following query? Query: {query}"
        )
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content="I need an answer based on the above information.",
            ),
        ]
        response = self.llm.chat(messages)
        cleaned_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return cleaned_response

    def aggregate_answers(self, community_answers):
        """Aggregate individual community answers into a final, coherent response."""
        # intermediate_text = " ".join(community_answers)
        prompt = "Combine the following intermediate answers into a final, concise response."
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content=f"Intermediate answers: {community_answers}",
            ),
        ]
        final_response = self.llm.chat(messages)
        cleaned_final_response = re.sub(
            r"^assistant:\s*", "", str(final_response)
        ).strip()
        return cleaned_final_response

##  Build End to End GraphRAG Pipeline

Now that we have defined all the necessary components, let’s construct the GraphRAG pipeline:

1. Create nodes/chunks from the text.
2. Build a PropertyGraphIndex using `GraphRAGExtractor` and `GraphRAGStore`.
3. Construct communities and generate a summary for each community using the graph built above.
4. Create a `GraphRAGQueryEngine` and begin querying.

### Create nodes/ chunks from the text.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=20,
)
nodes = splitter.get_nodes_from_documents(documents)

In [ ]:
len(nodes)

24

### Build ProperGraphIndex using `GraphRAGExtractor` and `GraphRAGStore`

In [ ]:
KG_TRIPLET_EXTRACT_TMPL = """
-Goal-
Given a text document, identify all entities and their entity types from the text and all relationships among the identified entities.
Given the text, extract up to {max_knowledge_triplets} entity-relation triplets.

-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: Type of the entity
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"$$$$<entity_name>$$$$<entity_type>$$$$<entity_description>)

2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relation: relationship between source_entity and target_entity
- relationship_description: explanation as to why you think the source entity and the target entity are related to each other

Format each relationship as ("relationship"$$$$<source_entity>$$$$<target_entity>$$$$<relation>$$$$<relationship_description>)

3. When finished, output.

-Real Data-
######################
text: {text}
######################
output:"""

In [ ]:
entity_pattern = r'\("entity"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'
relationship_pattern = r'\("relationship"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'


def parse_fn(response_str: str) -> Any:
    entities = re.findall(entity_pattern, response_str)
    relationships = re.findall(relationship_pattern, response_str)
    return entities, relationships


kg_extractor = GraphRAGExtractor(
    llm=llm,
    extract_prompt=KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk=2,
    parse_fn=parse_fn,
)

## Docker Setup And Neo4J setup

To launch Neo4j locally, first ensure you have docker installed. Then, you can launch the database with the following docker command.

```
docker run \
    -p 7474:7474 -p 7687:7687 \
    -v $PWD/data:/data -v $PWD/plugins:/plugins \
    --name neo4j-apoc \
    -e NEO4J_apoc_export_file_enabled=true \
    -e NEO4J_apoc_import_file_enabled=true \
    -e NEO4J_apoc_import_file_use__neo4j__config=true \
    -e NEO4JLABS_PLUGINS=\[\"apoc\"\] \
    neo4j:latest
```
From here, you can open the db at http://localhost:7474/. On this page, you will be asked to sign in. Use the default username/password of neo4j and neo4j.

Once you login for the first time, you will be asked to change the password.

In [ ]:
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

# Note: used to be `Neo4jPGStore`
graph_store = GraphRAGStore(
    username="neo4j", password="sFRfCfMDwSPGPc8-36wnN9NtvnCpTGbL-5ALEfXY6xQ", url="neo4j+s://0892bbb1.databases.neo4j.io"
)

In [ ]:
graph_store.structured_schema

{'node_props': {'Chunk': [{'property': 'id',
    'type': 'STRING',
    'values': ['00810167-9fb6-4fb2-ba32-40d7f12c56c6',
     'f9278d88-2ada-4b2f-8326-b9c42a2a8a5a',
     'f7a76220-f97c-4c2f-8107-df49eea38bc3',
     '56daf0ed-57ea-4132-8305-4a67b432dbb6',
     '757cfdc5-c05a-435b-bcd4-7bdb3e50a64f',
     'c20cbc5e-a85b-4511-b059-a3244ceeecf4',
     'd3839a7e-3b59-45a6-9d3b-ebb15534c870',
     '87cd2cca-59ae-4eaf-85b4-40a1d8dd243a',
     '0c372884-59f8-4401-83d1-e1b18011e2e0',
     'a633689e-8e87-4636-a50d-85faece1e6d8'],
    'distinct_count': 25},
   {'property': 'embedding',
    'type': 'EMBEDDING',
    'values': [-0.027747511863708496,
     0.03574371710419655,
     -0.03126150369644165],
    'max_size': 1024,
    'min_size': 0},
   {'property': 'text',
    'type': 'TEXT',
    'values': ['Original papers\nWho cares for the patient diabetes?\n',
     '1984, from practice morbidity registers (eight lists',
     'P.J. Burrows, P.J. Gray, A-L. Kinmonth, et al. Origi',
     '4M 6ea *n 1a

In [ ]:
from llama_index.core import PropertyGraphIndex

index = PropertyGraphIndex(
    nodes=nodes,
    kg_extractors=[kg_extractor],
    property_graph_store=graph_store,
    show_progress=True,
)

Generating embeddings: 100%|██████████| 3/3 [00:01<00:00,  2.50it/s]


In [ ]:
# index.property_graph_store.save_networkx_graph(name="./kg.html")

In [ ]:
graph_store.structured_schema

{'node_props': {'Chunk': [{'property': 'id',
    'type': 'STRING',
    'values': ['00810167-9fb6-4fb2-ba32-40d7f12c56c6',
     'f9278d88-2ada-4b2f-8326-b9c42a2a8a5a',
     'f7a76220-f97c-4c2f-8107-df49eea38bc3',
     '56daf0ed-57ea-4132-8305-4a67b432dbb6',
     '757cfdc5-c05a-435b-bcd4-7bdb3e50a64f',
     'c20cbc5e-a85b-4511-b059-a3244ceeecf4',
     'd3839a7e-3b59-45a6-9d3b-ebb15534c870',
     '87cd2cca-59ae-4eaf-85b4-40a1d8dd243a',
     '0c372884-59f8-4401-83d1-e1b18011e2e0',
     'a633689e-8e87-4636-a50d-85faece1e6d8'],
    'distinct_count': 25},
   {'property': 'embedding',
    'type': 'EMBEDDING',
    'values': [-0.027747511863708496,
     0.03574371710419655,
     -0.03126150369644165],
    'max_size': 1024,
    'min_size': 0},
   {'property': 'text',
    'type': 'TEXT',
    'values': ['Original papers\nWho cares for the patient diabetes?\n',
     '1984, from practice morbidity registers (eight lists',
     'P.J. Burrows, P.J. Gray, A-L. Kinmonth, et al. Origi',
     '4M 6ea *n 1a

In [ ]:
triplets = index.property_graph_store.get_triplets()
print("Total triplets:", len(triplets))



Total triplets: 10


In [ ]:
index.property_graph_store.get_triplets()[9]

[EntityNode(label='Behavior', embedding=None, properties={'id': 'PhysicalActivity', 'entity_description': 'Engaging in regular exercise or bodily movement for fitness or recreation', 'triplet_source_id': '3e184d5b-dd49-4e80-9256-3d41a4cbae81'}, name='PhysicalActivity'),
 Relation(label='Reduces', source_id='PhysicalActivity', target_id='RiskOfDiabetes', properties={'triplet_source_id': '3e184d5b-dd49-4e80-9256-3d41a4cbae81', 'relationship_description': 'Regular physical activity can lower the risk of developing type 2 diabetes.'}),
 EntityNode(label='Condition', embedding=None, properties={'id': 'RiskOfDiabetes', 'entity_description': 'Factors or conditions that increase the likelihood of developing type 2 diabetes', 'triplet_source_id': '3e184d5b-dd49-4e80-9256-3d41a4cbae81'}, name='RiskOfDiabetes')]

In [ ]:
index.property_graph_store.get_triplets()[9][0].properties

{'id': 'PhysicalActivity',
 'entity_description': 'Engaging in regular exercise or bodily movement for fitness or recreation',
 'triplet_source_id': '3e184d5b-dd49-4e80-9256-3d41a4cbae81'}

In [ ]:
retriever = index.as_retriever(
    include_text=False,  # include source text, default True
)

nodes = retriever.retrieve("What are type 1 and type 2 diabetes, and how to prevent from it?")

for node in nodes:
    print(node.text)

PhysicalActivity -> Reduces -> RiskOfDiabetes
WorkplaceResources -> Reduces -> RiskOfDiabetes
Diabetes -> Subtype -> Type 1 Diabetes
Diabetes -> Treatment -> Metformin


In [ ]:
query_engine = index.as_query_engine(
    include_text=True,
)

response = query_engine.query("What are type 1 and type 2 diabetes, and how to prevent from it?")

print(str(response))

: **Type 1 Diabetes**:
- **Definition**: Type 1 diabetes is a subtype of diabetes characterized by the body's immune system attacking and destroying the cells in the pancreas that produce insulin, leading to little or no insulin production. Insulin is essential for regulating blood sugar levels.
- **Prevention**: There is currently no known way to prevent type 1 diabetes since it is primarily caused by genetic factors and autoimmune responses.

**Type 2 Diabetes**:
- **Definition**: Type 2 diabetes is another form of diabetes where the body either does not produce enough insulin or cannot effectively use the insulin it produces (insulin resistance). This leads to elevated blood glucose levels over time.
- **Prevention**: 
  - **Physical Activity**: Regular exercise can help improve insulin sensitivity and control weight, which are crucial in preventing type 2 diabetes.
  - **Healthy Lifestyle Choices**: Maintaining a balanced diet rich in fruits, vegetables, whole grains, and lean prot

In [ ]:
from rich import print
print(response)

Response(
    response=": **Type 1 Diabetes**:\n- **Definition**: Type 1 diabetes is a subtype of diabetes characterized by 
the body's immune system attacking and destroying the cells in the pancreas that produce insulin, leading to little
or no insulin production. Insulin is essential for regulating blood sugar levels.\n- **Prevention**: There is 
currently no known way to prevent type 1 diabetes since it is primarily caused by genetic factors and autoimmune 
responses.\n\n**Type 2 Diabetes**:\n- **Definition**: Type 2 diabetes is another form of diabetes where the body 
either does not produce enough insulin or cannot effectively use the insulin it produces (insulin resistance). This
leads to elevated blood glucose levels over time.\n- **Prevention**: \n  - **Physical Activity**: Regular exercise 
can help improve insulin sensitivity and control weight, which are crucial in preventing type 2 diabetes.\n  - 
**Healthy Lifestyle Choices**: Maintaining a balanced diet rich in fruits, vegetables, whole grains, and lean 
proteins while limiting processed foods and sugars can significantly reduce the risk.\n  - **Weight Management**: 
Being overweight or obese increases the likelihood of developing type 2 diabetes. Maintaining a healthy weight 
through a combination of proper nutrition and regular physical activity is key.\n  - **Regular Medical Check-ups**:
Monitoring blood sugar levels and other health indicators can help detect early signs of diabetes and allow for 
timely interventions.\n\nIn summary, while type 1 diabetes cannot be prevented due to its autoimmune basis, 
adopting a healthier lifestyle with adequate physical activity, maintaining a balanced diet, managing weight, and 
undergoing regular medical check-ups can help prevent type 2 diabetes.",
    source_nodes=[
        NodeWithScore(
            node=TextNode(
                id_='a82d7c83-7dcd-453f-bec1-10eca8165bc8',
                embedding=None,
                metadata={},
                excluded_embed_metadata_keys=[],
                excluded_llm_metadata_keys=[],
                relationships={
                    <NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(
                        node_id='58d9c6ab-5a58-43f7-a2be-2a61f740cfa9',
                        node_type='4',
                        metadata={},
                        hash='d7c35db1055aa5a8e24ec3b47a29187a6d683c2bcbac8af0edd109308dfc94c4'
                    ),
                    <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(
                        node_id='0f588425-7c49-414e-abb9-c7b2bddf8f77',
                        node_type='1',
                        metadata={},
                        hash='2e8bcb0e6ce66983a7d347d84642b4bd10107504c79de80ce43c3e1810922f13'
                    )
                },
                metadata_template='{key}: {value}',
                metadata_separator='\n',
                text='Here are some facts extracted from the provided text:\n\nDiabetes -> Subtype -> Type 1 
Diabetes\nDiabetes -> Treatment -> Metformin\n\nwith diabetes development, absence of exposure 
measurementandallowed for\nOur study showed that health-related a multiplicative interaction implies the a 
virtually full follow-up. People <40\nlifestyle tends to differ according to presenceofanadditiveinteraction(37). 
years of age were excluded to reduce\nresource class. Stronger leadership at Additive interaction may be of better 
the possibility that taking metformin\nwork could make work-related lifestyle public health relevance because it 
esti- wasduetopolycysticovariansyndrome\ninterventions more successful (33). A matesthenumberofadditionaldiabetes 
and the possibility of being treated by\nlongitudinal association has been found cases prevented in one group 
compared insulin because of type 1 diabetes.\nbetween a high level of social capital 
withanother.Forolderemployees,work- Therefore, in our study, we believe to\nand an increased probability of smoking
place psychosocial res

In [ ]:
response.response

": **Type 1 Diabetes**:\n- **Definition**: Type 1 diabetes is a subtype of diabetes characterized by the body's immune system attacking and destroying the cells in the pancreas that produce insulin, leading to little or no insulin production. Insulin is essential for regulating blood sugar levels.\n- **Prevention**: There is currently no known way to prevent type 1 diabetes since it is primarily caused by genetic factors and autoimmune responses.\n\n**Type 2 Diabetes**:\n- **Definition**: Type 2 diabetes is another form of diabetes where the body either does not produce enough insulin or cannot effectively use the insulin it produces (insulin resistance). This leads to elevated blood glucose levels over time.\n- **Prevention**: \n  - **Physical Activity**: Regular exercise can help improve insulin sensitivity and control weight, which are crucial in preventing type 2 diabetes.\n  - **Healthy Lifestyle Choices**: Maintaining a balanced diet rich in fruits, vegetables, whole grains, and 

In [ ]:
# index.property_graph_store.get_triplets()[10][1].properties

### Build communities

This will create communities and summary for each community.

In [ ]:
# index.property_graph_store.build_communities()

### Create QueryEngine

In [ ]:
# query_engine = GraphRAGQueryEngine(
#     graph_store=index.property_graph_store,
#     llm=llm,
#     index=index,
#     similarity_top_k=10,
# )

### Querying

In [ ]:
# response = query_engine.query(
#     "What are the main news discussed in the document?"
# )
# display(Markdown(f"{response.response}"))

In [ ]:
# response = query_engine.query("What are the main news in energy sector?")
# display(Markdown(f"{response.response}"))

## **EVALUATION**

In [ ]:
import json
import pandas as pd
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator
)
from llama_index.llms.openai import OpenAI
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
# Load the evaluation dataset from the JSON file
def load_eval_dataset(json_file_path):
    with open(json_file_path, 'r') as f:
        data = [json.loads(line) for line in f.readlines()]
    return data

In [ ]:
!ollama list

/bin/bash: line 1: ollama: command not found


In [ ]:
from llama_index.llms.ollama import Ollama

# Initialize evaluation models
def init_evaluators():
    # Use GPT-4 for evaluation
    # gpt4 = OpenAI(temperature=0, model="gpt-4")
    eval_llm = Ollama(model="qwen2.5:14b", request_timeout=600.0)

    # Initialize evaluators
    faithfulness_evaluator = FaithfulnessEvaluator(llm=eval_llm)
    relevancy_evaluator = RelevancyEvaluator(llm=eval_llm)
    correctness_evaluator = CorrectnessEvaluator(llm=eval_llm)

    return faithfulness_evaluator, relevancy_evaluator, correctness_evaluator


In [ ]:
# # Function to run evaluations on a single query
# def evaluate_query(query_engine, query_data, evaluators):
#     query = query_data["question"]
#     reference = query_data["answer"]
#     context = query_data["context"]

#     # Get response from query engine
#     response = query_engine.query(query)

#     # Unpack evaluators
#     faithfulness_evaluator, relevancy_evaluator, correctness_evaluator = evaluators

#     # Evaluate faithfulness
#     faith_result = faithfulness_evaluator.evaluate_response(response=response)

#     # Evaluate relevancy of response to query
#     relevancy_result = relevancy_evaluator.evaluate_response(query=query, response=response)

#     # Evaluate relevancy of each source node if available
#     source_relevancy_results = []
#     if hasattr(response, 'source_nodes') and response.source_nodes:
#         for source_node in response.source_nodes:
#             node_result = relevancy_evaluator.evaluate(
#                 query=query,
#                 response=str(response),
#                 contexts=[source_node.get_content()]
#             )
#             source_relevancy_results.append(node_result.passing)

#         # Calculate percentage of relevant sources
#         source_relevancy_score = sum(source_relevancy_results) / len(source_relevancy_results)
#     else:
#         source_relevancy_score = None

#     # Evaluate correctness against reference answer
#     correctness_result = correctness_evaluator.evaluate(
#         query=query,
#         response=str(response),
#         reference=reference
#     )

#     # Return all evaluation results
#     return {
#         "query": query,
#         "response": str(response),
#         "reference": reference,
#         "faithfulness_score": 1 if faith_result.passing else 0,
#         "faithfulness_feedback": faith_result.feedback,
#         "relevancy_score": 1 if relevancy_result.passing else 0,
#         "relevancy_feedback": relevancy_result.feedback,
#         "source_relevancy_score": source_relevancy_score,
#         "correctness_score": correctness_result.score,
#         "correctness_passing": 1 if correctness_result.passing else 0,
#         "correctness_feedback": correctness_result.feedback
#     }

In [ ]:
# Function to run evaluations on a single query
def evaluate_query(query_engine, query_data, evaluators):
    query = query_data["question"]
    reference = query_data["answer"]
    context = query_data["context"]

    # Get response from query engine
    response = query_engine.query(query)

    # Extract the response text from the Response object
    # This handles the specific format of your query engine's response
    if hasattr(response, 'response'):
        response_text = response.response
    else:
        response_text = str(response)

    # Unpack evaluators
    faithfulness_evaluator, relevancy_evaluator, correctness_evaluator = evaluators

    # Evaluate faithfulness - pass the full Response object
    # LlamaIndex evaluators can handle the Response object directly
    faith_result = faithfulness_evaluator.evaluate_response(response=response)

    # Evaluate relevancy of response to query
    # For relevancy, we need to explicitly pass the text content
    relevancy_result = relevancy_evaluator.evaluate(
        query=query,
        response=response_text,
        contexts=[context]  # Use the context from the dataset
    )

    # Evaluate relevancy of each source node if available
    source_relevancy_results = []
    if hasattr(response, 'source_nodes') and response.source_nodes:
        for source_node in response.source_nodes:
            # Extract source content properly
            if hasattr(source_node, 'node') and hasattr(source_node.node, 'text'):
                source_content = source_node.node.text
            else:
                source_content = str(source_node)

            node_result = relevancy_evaluator.evaluate(
                query=query,
                response=response_text,
                contexts=[source_content]
            )
            source_relevancy_results.append(node_result.passing)

        # Calculate percentage of relevant sources
        source_relevancy_score = sum(source_relevancy_results) / len(source_relevancy_results)
    else:
        source_relevancy_score = None

    # Evaluate correctness against reference answer
    correctness_result = correctness_evaluator.evaluate(
        query=query,
        response=response_text,
        reference=reference
    )

    # Return all evaluation results
    return {
        "query": query,
        "response": response_text,
        "reference": reference,
        "faithfulness_score": 1 if faith_result.passing else 0,
        "faithfulness_feedback": faith_result.feedback,
        "relevancy_score": 1 if relevancy_result.passing else 0,
        "relevancy_feedback": relevancy_result.feedback,
        "source_relevancy_score": source_relevancy_score,
        "correctness_score": correctness_result.score,
        "correctness_passing": 1 if correctness_result.passing else 0,
        "correctness_feedback": correctness_result.feedback
    }

In [ ]:
# Function to create visualizations
def create_visualizations(results_df, metrics):
    # Create a bar chart for metrics
    plt.figure(figsize=(10, 6))
    bars = plt.bar(list(metrics.keys()), list(metrics.values()))
    plt.ylim(0, 1.1)
    plt.title('RAG System Evaluation Metrics')
    plt.ylabel('Score')
    plt.xticks(rotation=45)

    # Add values on top of bars
    for bar in bars:
        height = bar.get_height()
        if height is not None:  # Handle None values
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                    f'{height:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.savefig('rag_evaluation_metrics.png')
    plt.close()

    # Create detailed metric breakdowns if needed
    if 'correctness_score' in results_df.columns:
        plt.figure(figsize=(10, 6))
        plt.hist(results_df['correctness_score'], bins=10, alpha=0.7)
        plt.title('Distribution of Correctness Scores')
        plt.xlabel('Score')
        plt.ylabel('Count')
        plt.savefig('correctness_distribution.png')
        plt.close()


In [ ]:
# Main evaluation function
def evaluate_rag_system(query_engine, dataset_path):
    # Load dataset
    eval_dataset = load_eval_dataset(dataset_path)
    print(f"Loaded {len(eval_dataset)} evaluation samples")

    # Initialize evaluators
    evaluators = init_evaluators()

    # Run evaluations for each query in the dataset
    results = []
    for query_data in tqdm(eval_dataset, desc="Evaluating queries"):
        try:
            result = evaluate_query(query_engine, query_data, evaluators)
            results.append(result)
        except Exception as e:
            print(f"Error evaluating query '{query_data['question']}': {str(e)}")

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Calculate aggregate metrics
    metrics = {
        "Faithfulness": results_df["faithfulness_score"].mean(),
        "Relevancy": results_df["relevancy_score"].mean(),
        "Source Relevancy": results_df["source_relevancy_score"].dropna().mean() if not results_df["source_relevancy_score"].dropna().empty else None,
        "Correctness Score": results_df["correctness_score"].mean(),
        "Correctness Pass Rate": results_df["correctness_passing"].mean()
    }

    # Create a metrics summary DataFrame
    metrics_df = pd.DataFrame({
        "Metric": list(metrics.keys()),
        "Score": list(metrics.values())
    })

    # Generate visualizations
    create_visualizations(results_df, metrics)

    return results_df, metrics_df

In [ ]:
# Main execution function for the RAG evaluation
def run_rag_evaluation(query_engine, dataset_path="./mini_medical_qa_dataset.jsonl"):
    print("Starting RAG system evaluation...")

    # Run the evaluation
    results_df, metrics_df = evaluate_rag_system(query_engine, dataset_path)

    # Save detailed results to CSV
    results_df.to_csv("rag_evaluation_detailed_results.csv", index=False)

    # Display the metrics summary
    print("\nRAG Evaluation Summary:")
    print(metrics_df.to_string(index=False))

    print("\nDetailed results saved to 'rag_evaluation_detailed_results.csv'")
    print("Visualizations saved as PNG files")

    return results_df, metrics_df

In [ ]:
# Example usage (uncomment to run)
if __name__ == "__main__":
    # Assuming your query_engine and dataset are ready
    results_df, metrics_df = run_rag_evaluation(query_engine, "./mini_medical_qa_dataset.jsonl")


Starting RAG system evaluation...

Loaded 20 evaluation samples

Evaluating queries: 100%|██████████| 20/20 [11:45<00:00, 35.30s/it]


RAG Evaluation Summary:

Metric    Score
         Faithfulness 0.000000
            Relevancy 0.150000
     Source Relevancy 0.361111
    Correctness Score 1.050000
Correctness Pass Rate 0.000000

Detailed results saved to 'rag_evaluation_detailed_results.csv'

Visualizations saved as PNG files

In [ ]:
metrics_df

,Metric,Score
0,Faithfulness,0.000000
1,Relevancy,0.150000
2,Source Relevancy,0.361111
3,Correctness Score,1.050000
4,Correctness Pass Rate,0.000000


In [ ]:
metrics_df

,Metric,Score
0,Faithfulness,0.00
1,Relevancy,0.15
2,Source Relevancy,0.30
3,Correctness Score,1.15
4,Correctness Pass Rate,0.05


In [ ]:
results_df

,query,response,reference,faithfulness_score,faithfulness_feedback,relevancy_score,relevancy_feedback,source_relevancy_score,correctness_score,correctness_passing,correctness_feedback
0,What is the primary factor that predicts outco...,":\nGiven the new context provided, it is still...",Self-management competence,0,NO\n\nThe provided text does not contain infor...,1,"YES\n\nThe response provided, while not direct...",0.00,2.0,0,The generated answer provides a detailed expla...
1,What is the relationship between stress and me...,Empty Response,Social competence and parental support are med...,0,Empty Response,0,"NO\n\nThe response is empty, but the context p...",NaN,1.0,0,The generated answer does not provide any rele...
2,What is a key factor that influences adherence...,Empty Response,Psychosocial variables and metabolic control.,0,Empty Response,0,"NO\n\nThe response to the query ""What is a key...",NaN,1.0,0,The generated answer does not provide any info...
3,What is the mean raw score for youths ≥11 year...,": Based on the provided context, there is stil...",21.9 ± 3.6,0,NO\n\nThe provided context discusses Paul Grah...,0,NO\n\nThe context provided includes data relat...,0.00,1.0,0,The generated answer is completely irrelevant ...
4,What is the mean raw score for youths ≥11 year...,: The provided context does not contain any in...,12.4 ± 2.9,0,NO,0,NO\n\nThe context provided does actually conta...,1.00,1.0,0,The generated answer is not relevant because i...
5,What is the mean maximum score for youths aged...,: The provided context does not contain any in...,12,0,NO,0,NO\n\nThe response correctly states that the p...,1.00,1.0,0,The generated answer is completely irrelevant ...
6,What is the mean adherence score of the DSMP-F...,: The information provided does not contain an...,86,0,NO,0,NO\n\nThe response provided does not align wit...,0.00,1.0,0,The generated answer is not relevant to the us...
7,What is the mean adherence score of the DSMP-F...,": Based on the provided context, there is stil...",73% of the maximum DSMP total score (86),0,NO\n\nThe additional text provided does not co...,1,YES\n\nThe response accurately reflects that t...,1.00,1.0,0,The generated answer is entirely irrelevant to...
8,What is the mean adherence score of parents an...,:\nThe provided context does not contain any i...,73% of the maximum DSMP total score (86),0,NO\n\nThe additional text provided does not co...,1,YES\n\nThe response acknowledges that the cont...,0.25,1.0,0,The generated answer is not relevant because i...
9,What is the name of the modified structured in...,Empty Response,Diabetes Self-Management Profile for Flexible ...,0,Empty Response,0,"NO\n\nThe response to the query is empty, whil...",NaN,1.0,0,The generated answer is completely empty and d...
